In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Imports successful!")
print(f"🖥️  Using device: {device}")
print(f"🔥 PyTorch version: {torch.__version__}")

✅ Imports successful!
🖥️  Using device: cuda
🔥 PyTorch version: 2.6.0+cu124


In [4]:
columns = ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + \
          [f'sensor_{i}' for i in range(1, 22)]

train_df = pd.read_csv('../data/raw/train_FD001.txt', sep=r'\s+', header=None, names=columns)
test_df  = pd.read_csv('../data/raw/test_FD001.txt',  sep=r'\s+', header=None, names=columns)

# Drop dead + weak sensors
drop_sensors = ['sensor_1','sensor_5','sensor_6','sensor_10',
                'sensor_16','sensor_18','sensor_19','sensor_8','sensor_13','sensor_15']
train_df.drop(columns=drop_sensors, inplace=True)
test_df.drop(columns=drop_sensors, inplace=True)

# Add RUL
max_cycles = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='engine_id', how='left')
train_df['RUL'] = (train_df['max_cycle'] - train_df['cycle']).clip(upper=125)
train_df.drop(columns=['max_cycle'], inplace=True)

feature_cols = ['cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
                'sensor_2','sensor_3','sensor_4','sensor_7','sensor_9',
                'sensor_11','sensor_12','sensor_14','sensor_17','sensor_20','sensor_21']

print(f"✅ Data loaded! Shape: {train_df.shape}")

✅ Data loaded! Shape: (20631, 17)


In [5]:
WINDOW_SIZE = 30

# Normalize features
scaler = MinMaxScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])

# Create sliding windows per engine
def create_windows(df, window_size, feature_cols):
    X, y = [], []
    for eng_id in df['engine_id'].unique():
        eng_data = df[df['engine_id'] == eng_id].reset_index(drop=True)
        for i in range(len(eng_data) - window_size + 1):
            X.append(eng_data[feature_cols].iloc[i:i+window_size].values)
            y.append(eng_data['RUL'].iloc[i+window_size-1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X, y = create_windows(train_df, WINDOW_SIZE, feature_cols)

# Train/val split by engine
from sklearn.model_selection import GroupShuffleSplit
engine_ids = []
for eng_id in train_df['engine_id'].unique():
    eng_data = train_df[train_df['engine_id'] == eng_id]
    engine_ids.extend([eng_id] * max(0, len(eng_data) - WINDOW_SIZE + 1))
engine_ids = np.array(engine_ids)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, groups=engine_ids))

X_train, X_val = X[train_idx], X[val_idx]
y_train, y_val = y[train_idx], y[val_idx]

print(f"✅ Windows created!")
print(f"X_train: {X_train.shape} → (samples, window, features)")
print(f"X_val  : {X_val.shape}")

✅ Windows created!
X_train: (14241, 30, 15) → (samples, window, features)
X_val  : (3490, 30, 15)


In [10]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
best_val_loss = float('inf')
train_losses, val_losses = [], []

print("🚀 Training LSTM on GPU...")
for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            pred = model(X_batch)
            val_loss += criterion(pred, y_batch).item()

    train_loss /= len(train_loader)
    val_loss   /= len(val_loader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../models/lstm_best.pt')

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train Loss: {train_loss:.2f} | Val Loss: {val_loss:.2f}")

print(f"\n✅ Training complete! Best Val Loss: {best_val_loss:.2f}")

🚀 Training LSTM on GPU...
Epoch  10/50 | Train Loss: 262.00 | Val Loss: 175.86
Epoch  20/50 | Train Loss: 228.80 | Val Loss: 162.40
Epoch  30/50 | Train Loss: 210.42 | Val Loss: 178.14
Epoch  40/50 | Train Loss: 199.90 | Val Loss: 194.73
Epoch  50/50 | Train Loss: 191.97 | Val Loss: 191.59

✅ Training complete! Best Val Loss: 159.02


In [8]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.fc(last_out).squeeze()

model = LSTMModel(input_size=len(feature_cols)).to(device)
print(f"✅ LSTM Model created!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)

✅ LSTM Model created!
Parameters: 214,657
LSTMModel(
  (lstm): LSTM(15, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [11]:
model.load_state_dict(torch.load('../models/lstm_best.pt'))
model.eval()

y_pred_list = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch)
        y_pred_list.extend(pred.cpu().numpy())

y_pred = np.array(y_pred_list)
y_pred = np.clip(y_pred, 0, 125)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae  = mean_absolute_error(y_val, y_pred)

print(f"📊 LSTM Results:")
print(f"   RMSE : {rmse:.2f} cycles")
print(f"   MAE  : {mae:.2f} cycles")
print(f"\n📊 Comparison:")
print(f"   RF   RMSE : 15.37 cycles")
print(f"   LSTM RMSE : {rmse:.2f} cycles")
if rmse < 15.37:
    print(f"   ✅ LSTM beats RF by {15.37-rmse:.2f} cycles!")
else:
    print(f"   ❌ RF still better — need tuning")

📊 LSTM Results:
   RMSE : 12.64 cycles
   MAE  : 9.32 cycles

📊 Comparison:
   RF   RMSE : 15.37 cycles
   LSTM RMSE : 12.64 cycles
   ✅ LSTM beats RF by 2.73 cycles!
